## self attention using trainable weights

In [1]:
import numpy as np
import torch
import math

### get the inout token embeddings

In [2]:
# input embeddings
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your 
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]  # step
])

### Linear projection

In [3]:
# take one embedding from implementation
x_2 = inputs[1]

# input dimension
dimension_input = inputs.shape[1]
dimension_input

# output dimension
dimension_output = 2

In [5]:
# set the seed value to get same output every time
torch.manual_seed(123)

# create the weight matrices for Q, K, V 
weight_query = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)
weight_key = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)
weight_value = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)


In [6]:
weight_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])

In [7]:
weight_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])

In [8]:
weight_value

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])

## Compute the Q,K,V for second embedding

In [9]:
x_2

tensor([0.5500, 0.8700, 0.6600])

In [11]:
# compute the Q(Query) - the embedding used to find the attention wrt all the embeddings
query_2 = x_2 @ weight_query
query_2

tensor([0.4306, 1.4551])

In [13]:
# compute the K(Key) - the embedding with which relevance will be calculated
key_2 = x_2 @ weight_key
key_2

tensor([0.4433, 1.1419])

In [15]:
# Compute the V (value) - the actual content
value_2 = x_2 @ weight_key
value_2

tensor([0.4433, 1.1419])

## compute Q, K, V for all the embeddings

In [16]:
queries = inputs @ weight_query
queries

tensor([[0.2309, 1.0966],
        [0.4306, 1.4551],
        [0.4300, 1.4343],
        [0.2355, 0.7990],
        [0.2983, 0.6565],
        [0.2568, 1.0533]])

In [17]:
keys = inputs @ weight_key
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])

In [18]:
values = inputs @ weight_value
values

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])

## computes attention scores

In [19]:
# find the attention scores only for 'journey'
attention_scores_journey = query_2 @ keys.T
attention_scores_journey

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

In [21]:
attention_scores = queries @ keys.T
attention_scores

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])

### calculate the scaled weights
- used to stabilize the training
- reducing the variance in the data
- bring the attentions scores near 1

In [ ]:
# get the dimension of keys
 # shape [Batch_Size, Sequence_Length, Embedding_Dimension].
dimension_keys = keys.shape[-1]
dimension_keys

torch.Size([6, 2])


2

In [43]:
# calculate the scaled weights
scaled_weights = attention_scores / math.sqrt(dimension_keys)
scaled_weights

tensor([[0.6528, 0.9578, 0.9363, 0.5593, 0.2851, 0.8011],
        [0.8984, 1.3098, 1.2806, 0.7633, 0.3944, 1.0918],
        [0.8870, 1.2929, 1.2641, 0.7534, 0.3895, 1.0775],
        [0.4930, 0.7189, 0.7029, 0.4190, 0.2164, 0.5993],
        [0.4323, 0.6236, 0.6099, 0.3621, 0.1914, 0.5167],
        [0.6361, 0.9309, 0.9101, 0.5432, 0.2784, 0.7776]])

### Calculate the attention weights using softmax

In [44]:
attention_weights = torch.softmax(scaled_weights, dim=-1)
attention_weights

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

## calculate the context vector
- Using weighted sum with values

In [46]:
context_vector = attention_weights @ values
context_vector

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])